# HealthyCrop — Phase 1 & 2: Coffee Disease Classification (`MobileNetV3-Large`)

This Google Colab notebook mirrors `ml/src/healthycrop_ml/train.py` (`ml/scripts/train_coffee.py`).

- **Crop**: Coffee (*Coffea arabica* & *Coffea canephora*)
- **Datasets**: RoCoLe (`10.17632/c5yvn32dzg.2`) + BRACOL (`10.17632/yy2k5y8mxg.1`, whole-leaf subset)
- **Classes (6)**: `healthy`, `leaf_rust`, `leaf_miner`, `cercospora`, `brown_leaf_spot`, `red_spider_mite`
- **Input Tensor**: `224x224x3` RGB normalized to `[-1.0, 1.0]` (`(rgb / 127.5) - 1.0`)
- **Stage 1 (Phase 1 Baseline)**: Frozen `MobileNetV3-Large` (`ImageNet`) backbone + classification head warm-up with inverse-frequency balanced class weights.
- **Stage 2 (Phase 2 Fine-Tuning)**: Unfreeze top backbone layers (keeping `BatchNormalization` frozen) at a lower learning rate with early stopping on validation `macro-F1`.

In [ ]:
# 1. Clone repo & install ml package in Colab
# !git clone https://github.com/keylorpineda/healthycrop.git
# %cd healthycrop/ml
# !pip install -e .

In [ ]:
# 2. Download and prepare leakage-free 70/15/15 coffee dataset
# !python scripts/download_rocole.py
# !python scripts/download_bracol.py
# !python scripts/prepare_coffee.py

In [ ]:
# 3. Phase 1: Train frozen-backbone MobileNetV3-Large baseline
from pathlib import Path

from healthycrop_ml.config import load_crop_config
from healthycrop_ml.train import run_training_pipeline

repo_root = Path("..").resolve()
cfg = load_crop_config(repo_root / "ml" / "configs" / "coffee.yaml")
processed_dir = repo_root / "ml" / "data" / "processed" / "coffee"

baseline_results = run_training_pipeline(
    cfg=cfg,
    processed_dir=processed_dir,
    run_id="coffee-v0.1.0-baseline",
    head_epochs=15,
    head_lr=1e-3,
    fine_tune_epochs=0,
    fine_tune_top_layers=0,
    batch_size=32,
    patience=4,
    weights="imagenet",
    checkpoint_out=repo_root / "ml" / "models" / "coffee" / "coffee_baseline.keras",
    confusion_png_out=repo_root / "docs" / "assets" / "confusion_matrices" / "coffee_baseline.png",
    metrics_json_out=processed_dir / "baseline_metrics.json",
)
print("Baseline Test Macro-F1:", baseline_results["test_metrics"]["macro_f1"])

In [ ]:
# 4. Phase 2: Fine-tune top layers of MobileNetV3-Large at lower learning rate
finetune_results = run_training_pipeline(
    cfg=cfg,
    processed_dir=processed_dir,
    run_id="coffee-v0.1.0-finetuned",
    head_epochs=12,
    head_lr=1e-3,
    fine_tune_epochs=25,
    fine_tune_lr=1e-4,
    fine_tune_top_layers=80,
    batch_size=32,
    patience=6,
    weights="imagenet",
    checkpoint_out=repo_root / "ml" / "models" / "coffee" / "coffee_finetuned.keras",
    confusion_png_out=repo_root / "docs" / "assets" / "confusion_matrices" / "coffee_finetuned.png",
    metrics_json_out=processed_dir / "finetune_metrics.json",
)
print("Fine-Tuned Test Macro-F1:", finetune_results["test_metrics"]["macro_f1"])